# Decision Trees: Zero to Hero

The only model here you can print out and hand to a doctor.

> **Prerequisite: [`ml_foundations_zero_to_hero.ipynb`](ml_foundations_zero_to_hero.ipynb).**
> Splitting, leakage, pipelines, messy data, metrics, tuning and shipping are covered there
> and are not repeated. This notebook covers only what is specific to decision trees.

---

## Why learn this one properly

A single decision tree is rarely the model you ship — it is usually beaten by a random forest
or a boosted ensemble. Learn it anyway, because:

- **It is the atom.** Random forests, gradient boosting, XGBoost and LightGBM are all
  *collections of these*. Every idea in NB-04 and NB-05 assumes this one.
- **It is genuinely readable.** Not "interpretable" in the hand-wavy sense — the model *is* a
  flowchart, and `export_text` prints it.
- **It behaves unlike everything else.** No scaling, no encoding of ordinals, native handling
  of interactions and non-linearity, and complete indifference to feature units. Knowing
  where that freedom ends is most of the skill.
- **Its weaknesses are instructive.** The instability in Part 3 is the single best motivation
  for ensembling, which is the whole of the next two notebooks.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Install + imports |
| **1. Theory from zero** | Recursive partitioning, impurity, building one from scratch, why greedy ≠ optimal, pruning, scale invariance, regression trees |
| **2. Worked example** | Breast cancer diagnosis — including printing the actual rules |
| **3. Instability** | The signature weakness, quantified — and why it leads directly to ensembles |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice datasets** | 5 datasets with briefs |
| **6. Reading the literature** | The papers behind each section |
| **Appendix** | Tree-specific errors and a checklist |

## The one-paragraph summary

A decision tree splits the data into two groups on one feature at a time, choosing the split
that most reduces **impurity**, then repeats on each group until a stopping rule fires. Every
split is chosen **greedily** — best right now, never looking ahead — because finding the
optimal tree is NP-hard. Left unchecked it will memorise the training set perfectly, so the
real work is controlling its size, via `max_depth` or **cost-complexity pruning**. Because it
only compares feature values to thresholds, it is completely invariant to any monotone
rescaling of the inputs.

---
# Part 0 - Setup

In [ ]:
# ---------------------------------------------------------------------------
# One-time setup. Only missing packages are installed, so re-running is cheap.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("scipy", "scipy"), ("sklearn", "scikit-learn"),
]
missing = [pip for mod, pip in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("done")
else:
    print("all packages present")

In [ ]:
# ---------------------------------------------------------------------------
# Every import this notebook uses.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import (
    DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree,
)
from sklearn.datasets import load_breast_cancer, load_wine, make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, mean_absolute_error,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110

print("ready | numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 - Theory from zero

1. The idea: twenty questions, played optimally
2. Choosing a split: impurity, Gini and entropy
3. Building one from scratch
4. Why greedy is not optimal — and why we do it anyway
5. Overfitting, and two ways to stop it
6. Scale invariance, and the price: axis-aligned boundaries
7. Regression trees

## 1.1 The idea: twenty questions

Every other model in this series computes a weighted sum. A tree does something completely
different: it asks a **sequence of yes/no questions**, each one narrowing the group the
sample belongs to, until it reaches a leaf that says "these are mostly class A".

Schematically — the real fitted tree is printed a few cells below:

```
                  [ all training patients ]
                     some_measure <= t1 ?
                    /                  \
                  yes                  no
                   /                    \
        [ mostly benign ]        [ mostly malignant ]
        another <= t2 ?                 ...
           /        \
         yes        no
          /          \
  [ almost all     [ mixed - split
     benign ]        again, or stop ]
```

Three consequences fall straight out of that structure, and they explain almost everything
about how trees behave:

1. **A split compares one feature to a threshold.** So only the *order* of the values
   matters, never their scale — hence 1.6.
2. **Each branch is fitted on a subset of the data.** So a tree learns interactions for free:
   the second question can differ depending on the answer to the first.
3. **Groups get smaller with depth.** By depth 10 a leaf may hold three patients, and a rule
   fitted to three patients is noise — hence 1.5.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_full, y_full = cancer.data, cancer.target      # 1 = benign, 0 = malignant
feature_names = list(X_full.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.25, stratify=y_full, random_state=RANDOM_STATE
)
print(f"{len(X_full)} patients, {X_full.shape[1]} measurements each")
print(f"target: {y_full.value_counts().to_dict()}   (1 = benign, 0 = malignant)")
print(f"train {len(X_train)}  test {len(X_test)}")

# A depth-2 tree, printed as the flowchart it literally is.
tiny = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(X_train, y_train)
print("\n" + export_text(tiny, feature_names=feature_names))
print(f"this 3-question model already scores {tiny.score(X_test, y_test):.4f} on the test set")
print()
print("Read it top to bottom: that IS the model. There is nothing else - no weights, no")
print("intercept, no transformation. Every prediction it will ever make is on that page.")

## 1.2 Choosing a split: impurity

At each node the tree tries **every feature × every threshold** and keeps the split that most
reduces *impurity* — a measure of how mixed the labels are.

**Gini impurity** — the probability that two randomly drawn members of the node have
different labels:

$$ G = 1 - \sum_k p_k^2 $$

**Entropy** — the information content, in bits:

$$ H = -\sum_k p_k \log_2 p_k $$

Both are 0 for a pure node and maximal for a 50/50 split (Gini 0.5, entropy 1.0 for two
classes). The tree scores a candidate split by the **weighted impurity of the children**, and
the reduction is the *information gain*:

$$ \text{gain} = I(\text{parent}) - \frac{n_L}{n}I(\text{left}) - \frac{n_R}{n}I(\text{right}) $$

In [ ]:
def gini(y):
    """Gini impurity of a set of labels."""
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=2) / len(y)
    return 1.0 - np.sum(p ** 2)


def entropy(y):
    """Shannon entropy, in bits."""
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=2) / len(y)
    p = p[p > 0]                                  # 0*log0 = 0 by convention
    return float(-np.sum(p * np.log2(p)) + 0.0)   # + 0.0 avoids printing "-0.0000"


print(f"{'class mix':<22} {'Gini':>8} {'entropy':>9}")
print("-" * 42)
for label, arr in [("100% one class", np.zeros(10, int)),
                   ("90 / 10", np.array([0]*9 + [1])),
                   ("70 / 30", np.array([0]*7 + [1]*3)),
                   ("50 / 50 (worst)", np.array([0]*5 + [1]*5))]:
    print(f"{label:<22} {gini(arr):>8.4f} {entropy(arr):>9.4f}")

# Both peak at 50/50 and hit zero when pure - they measure the same thing on
# different scales.
p_axis = np.linspace(0.001, 0.999, 300)
plt.plot(p_axis, 1 - (p_axis**2 + (1-p_axis)**2), label="Gini")
plt.plot(p_axis, -(p_axis*np.log2(p_axis) + (1-p_axis)*np.log2(1-p_axis)), label="entropy")
plt.xlabel("proportion of class 1 in the node"); plt.ylabel("impurity")
plt.title("Two impurity measures, the same shape")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Find the single best split by brute force, exactly as the algorithm does.
def best_split(X, y, impurity=gini):
    """Search every feature and every midpoint threshold. Returns (gain, feature, threshold)."""
    n, n_features = X.shape
    parent = impurity(y)
    best = (0.0, None, None)

    for f in range(n_features):
        values = np.unique(X[:, f])
        if len(values) < 2:
            continue
        # Candidate thresholds are midpoints between consecutive distinct values.
        for t in (values[:-1] + values[1:]) / 2:
            mask = X[:, f] <= t
            n_left = mask.sum()
            if n_left == 0 or n_left == n:
                continue
            weighted = (n_left / n) * impurity(y[mask]) + \
                       ((n - n_left) / n) * impurity(y[~mask])
            if parent - weighted > best[0]:
                best = (parent - weighted, f, t)
    return best


# 10 of the 30 features, to keep the brute-force search quick.
subset = list(range(0, 30, 3))
Xtr_np, Xte_np = X_train.to_numpy()[:, subset], X_test.to_numpy()[:, subset]
ytr_np, yte_np = y_train.to_numpy(), y_test.to_numpy()
subset_names = [feature_names[i] for i in subset]

gain, f, t = best_split(Xtr_np, ytr_np)
print(f"parent Gini: {gini(ytr_np):.4f}")
print(f"best split : '{subset_names[f]}' <= {t:.4f}")
print(f"gain       : {gain:.4f}   -> weighted child Gini {gini(ytr_np) - gain:.4f}")

sk_root = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE).fit(Xtr_np, ytr_np)
print(f"\nsklearn picks: feature {sk_root.tree_.feature[0]} "
      f"('{subset_names[sk_root.tree_.feature[0]]}') <= {sk_root.tree_.threshold[0]:.4f}")
print("Same split. That is the entire split-selection algorithm - exhaustive search.")

### Gini or entropy? It almost never matters

This gets asked in interviews far more than it matters in practice. Both peak at 50/50 and
vanish at purity, and their disagreements are confined to a narrow band of class ratios.

Rather than assert it, measure it two ways: with error bars, and by how often they choose the
same split.

In [ ]:
repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=RANDOM_STATE)

print("accuracy over 30 folds (depth-4 trees):")
for criterion in ["gini", "entropy"]:
    s = cross_val_score(
        DecisionTreeClassifier(criterion=criterion, max_depth=4, random_state=RANDOM_STATE),
        X_full, y_full, cv=repeated)
    print(f"  {criterion:<9} {s.mean():.4f} +/- {s.std():.4f}")

# Do they pick the same ROOT split? Test across 100 bootstrap resamples.
rng = np.random.default_rng(0)
agree = 0
for _ in range(100):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    Xb, yb = X_train.to_numpy()[idx], y_train.to_numpy()[idx]
    g = DecisionTreeClassifier(criterion="gini", max_depth=3,
                               random_state=RANDOM_STATE).fit(Xb, yb)
    e = DecisionTreeClassifier(criterion="entropy", max_depth=3,
                               random_state=RANDOM_STATE).fit(Xb, yb)
    agree += int(g.tree_.feature[0] == e.tree_.feature[0])

print(f"\nsame root feature in {agree}/100 bootstrap resamples")
print()
print("The accuracy difference is a small fraction of one standard deviation, and they")
print("agree on the root split the large majority of the time. This is not a hyperparameter")
print("worth tuning.")
print()
print("The real reason Gini is the default: entropy needs a logarithm per candidate split,")
print("Gini needs a multiply. On millions of candidate splits that adds up, and the trees")
print("come out nearly the same anyway.")

## 1.3 Building one from scratch

We have the split-finder. A tree is just that, applied recursively, with a stopping rule.
That is the entire CART algorithm — about twenty lines.

In [ ]:
def build_tree(X, y, depth=0, max_depth=3, min_samples_split=2):
    """Grow a classification tree recursively. Returns a nested dict."""
    # --- stopping rules ---
    if (depth >= max_depth
            or len(np.unique(y)) == 1              # already pure
            or len(y) < min_samples_split):
        return {"leaf": True, "prediction": int(np.bincount(y, minlength=2).argmax()),
                "n": len(y), "gini": gini(y)}

    gain, feature, threshold = best_split(X, y)
    if feature is None or gain <= 1e-12:           # no split helps
        return {"leaf": True, "prediction": int(np.bincount(y, minlength=2).argmax()),
                "n": len(y), "gini": gini(y)}

    mask = X[:, feature] <= threshold
    return {
        "leaf": False, "feature": feature, "threshold": threshold, "gain": gain,
        "n": len(y),
        "left":  build_tree(X[mask],  y[mask],  depth + 1, max_depth, min_samples_split),
        "right": build_tree(X[~mask], y[~mask], depth + 1, max_depth, min_samples_split),
    }


def predict_one(node, x):
    """Walk the tree until a leaf."""
    while not node["leaf"]:
        node = node["left"] if x[node["feature"]] <= node["threshold"] else node["right"]
    return node["prediction"]


def print_tree(node, names, indent="", branch=""):
    if node["leaf"]:
        print(f"{indent}{branch}-> class {node['prediction']}  "
              f"(n={node['n']}, gini={node['gini']:.3f})")
        return
    print(f"{indent}{branch}[{names[node['feature']]} <= {node['threshold']:.4f}]  "
          f"gain={node['gain']:.4f}")
    print_tree(node["left"], names, indent + "   ", "yes: ")
    print_tree(node["right"], names, indent + "   ", "no : ")


my_tree = build_tree(Xtr_np, ytr_np, max_depth=3)
my_pred = np.array([predict_one(my_tree, x) for x in Xte_np])

sk_tree = DecisionTreeClassifier(max_depth=3, criterion="gini",
                                 random_state=RANDOM_STATE).fit(Xtr_np, ytr_np)

print(f"from scratch test accuracy: {(my_pred == yte_np).mean():.4f}")
print(f"scikit-learn test accuracy: {sk_tree.score(Xte_np, yte_np):.4f}")
print(f"predictions identical     : {np.array_equal(my_pred, sk_tree.predict(Xte_np))}")
print("\nour tree:")
print_tree(my_tree, subset_names)

Identical predictions, from twenty lines. There is no hidden cleverness in a decision tree —
the whole algorithm is *exhaustive search for one split, then recurse*.

What sklearn adds is engineering, not concept: sorted feature arrays so candidate thresholds
are evaluated in one pass instead of re-scanning, sparse support, missing-value handling, and
a Cython inner loop.

## 1.4 Greedy is not optimal — and we do it anyway

Notice what `best_split` does **not** do: consider what happens *after* the split. It takes
the best single split available right now, and it never reconsiders.

That is a real limitation, not a technicality. Here is the classic case where one-step
lookahead sees nothing at all.

In [ ]:
# XOR: y = 1 when exactly one of x1, x2 is 1.
rng = np.random.default_rng(0)
n_xor = 1200
X_xor = rng.integers(0, 2, size=(n_xor, 2)).astype(float)
y_xor = (X_xor[:, 0].astype(int) ^ X_xor[:, 1].astype(int))

print(f"parent Gini: {gini(y_xor):.4f}   (a 50/50 mix, maximally impure)\n")
print("what every possible FIRST split achieves:")
for f in range(2):
    mask = X_xor[:, f] <= 0.5
    weighted = (mask.sum()/n_xor) * gini(y_xor[mask]) + \
               ((~mask).sum()/n_xor) * gini(y_xor[~mask])
    print(f"  split on x{f}: weighted child Gini {weighted:.4f}   gain {gini(y_xor)-weighted:.6f}")

print("\nBoth gains are ~0. A greedy chooser sees no reason to prefer either split, and no")
print("reason to split at all. And yet:\n")
for depth in [1, 2, 3]:
    acc = DecisionTreeClassifier(max_depth=depth,
                                 random_state=RANDOM_STATE).fit(X_xor, y_xor).score(X_xor, y_xor)
    print(f"  tree of depth {depth}: accuracy {acc:.4f}")

print()
print("Depth 1 is chance. Depth 2 is PERFECT. The first split is worthless on its own and")
print("indispensable in combination - exactly the situation greedy search cannot reason about.")
print()
print("It works here only because sklearn splits anyway on a near-zero gain, and the second")
print("level then finds the structure. Add noise features and the tie is broken arbitrarily,")
print("and the tree can miss XOR entirely.")

### So why is every implementation greedy?

Because **finding the optimal decision tree is NP-hard** (Hyafil & Rivest, 1976). The number
of possible trees grows explosively with features and thresholds; exhaustive search is
hopeless beyond toy sizes.

Greedy growth is $O(n \cdot p \cdot \log n)$ per level and works well enough in practice.
Where it fails, the standard fixes are:

- **grow deeper and prune back** (1.5) — a form of delayed correction
- **ensembles** — many greedy trees on different data, averaged. This is the entire
  motivation for Part 3 and for NB-04
- **optimal-tree solvers** (MIP/SAT based, e.g. `pystreed`, GOSDT) — genuinely optimal for
  small, shallow trees, and a real option when you need a certified-minimal model for audit

That XOR failure is not a curiosity. It is the reason a single tree is rarely the final
answer, and the reason the next two notebooks exist.

## 1.5 Overfitting, and two ways to stop it

An unconstrained tree keeps splitting until every leaf is pure. On data with no duplicate
rows carrying conflicting labels, it **always reaches 100% training accuracy** — it can
always isolate one more sample. That is not learning; it is memorisation with extra steps.

In [ ]:
print(f"{'max_depth':>10} {'leaves':>8} {'train':>8} {'test':>8} {'gap':>8}")
print("-" * 46)
for depth in [1, 2, 3, 4, 5, 7, 10, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    tr, te = t.score(X_train, y_train), t.score(X_test, y_test)
    print(f"{str(depth):>10} {t.get_n_leaves():>8} {tr:>8.4f} {te:>8.4f} {tr-te:>+8.4f}")

print()
print("Training accuracy climbs to exactly 1.0000 and stays there. Test accuracy peaks")
print("early and then DECLINES - the extra depth is fitting noise, and the widening gap")
print("is the amount of noise memorised.")
print()
print("Note the deepest tree is not the worst by much here; breast cancer is an easy,")
print("low-noise dataset. On noisy data the decline is far steeper.")

### Two families of control

**Pre-pruning (early stopping)** — refuse to grow. Cheap, but you are guessing the right
limit in advance, and a split that looks useless now may enable a great one below it (the
XOR problem again).

| Parameter | Meaning | Typical |
|---|---|---|
| `max_depth` | Hard depth cap | 3–10 |
| `min_samples_leaf` | A leaf must hold at least this many | 1–50; **the most useful one** |
| `min_samples_split` | Do not split a node smaller than this | 2–50 |
| `max_leaf_nodes` | Total leaf budget, grown best-first | 10–100 |
| `min_impurity_decrease` | Reject splits gaining less than this | 0.0–0.01 |

**Post-pruning (cost-complexity pruning)** — grow the tree out, then cut back. This is
CART's original method and it is principled: minimise

$$ R_\alpha(T) = R(T) + \alpha\,|T| $$

where $R(T)$ is the tree's error and $|T|$ is its number of leaves. Raising $\alpha$ makes
leaves progressively more expensive. Crucially, this generates a **finite nested sequence**
of optimal subtrees — you then pick among them by cross-validation.

`cost_complexity_pruning_path` returns exactly those $\alpha$ values.

In [ ]:
path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(
    X_train, y_train)
alphas = np.unique(path.ccp_alphas[:-1])          # drop the last: it collapses to one leaf

print(f"the pruning path has {len(alphas)} distinct alphas, "
      f"from {alphas.min():.6f} to {alphas.max():.6f}")
print("(a FINITE sequence - alpha is continuous, but only these values change the tree)\n")

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for a in alphas:
    t = DecisionTreeClassifier(ccp_alpha=a, random_state=RANDOM_STATE).fit(X_train, y_train)
    scores = cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=RANDOM_STATE),
                             X_train, y_train, cv=cv)
    rows.append({"ccp_alpha": a, "leaves": t.get_n_leaves(),
                 "train": t.score(X_train, y_train),
                 "cv_mean": scores.mean(), "cv_std": scores.std()})
pruning = pd.DataFrame(rows)
best_alpha = float(pruning.loc[pruning["cv_mean"].idxmax(), "ccp_alpha"])

print(pruning.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(pruning["ccp_alpha"], pruning["train"], marker="o", ms=3, label="train")
axes[0].plot(pruning["ccp_alpha"], pruning["cv_mean"], marker="o", ms=3, label="CV mean")
axes[0].fill_between(pruning["ccp_alpha"], pruning["cv_mean"] - pruning["cv_std"],
                     pruning["cv_mean"] + pruning["cv_std"], alpha=0.2)
axes[0].axvline(best_alpha, ls="--", color="crimson", label=f"chosen {best_alpha:.4f}")
axes[0].set(xlabel="ccp_alpha", ylabel="accuracy", title="Pruning chosen inside training data")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(pruning["ccp_alpha"], pruning["leaves"], marker="o", ms=3, color="seagreen")
axes[1].set(xlabel="ccp_alpha", ylabel="number of leaves", title="Tree size collapses as alpha rises")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=RANDOM_STATE).fit(
    X_train, y_train)
full = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
full_acc, pruned_acc = full.score(X_test, y_test), pruned.score(X_test, y_test)
n_test = len(y_test)

print(f"\n{'model':<12} {'leaves':>7} {'CV (train data)':>17} {'test accuracy':>15}")
print("-" * 56)
print(f"{'unpruned':<12} {full.get_n_leaves():>7} "
      f"{pruning['cv_mean'].iloc[0]:>17.4f} {full_acc:>15.4f}")
print(f"{'pruned':<12} {pruned.get_n_leaves():>7} "
      f"{pruning['cv_mean'].max():>17.4f} {pruned_acc:>15.4f}")

print()
print(f"Cross-validation preferred the pruned tree ({pruning['cv_mean'].max():.4f} vs "
      f"{pruning['cv_mean'].iloc[0]:.4f}),")
print(f"but on this particular test set the unpruned tree came out {full_acc:.4f} to "
      f"{pruned_acc:.4f}.")
print()
print("Do NOT read that as 'pruning did not work'. Read the size of it: on "
      f"{n_test} test rows,")
print(f"that gap is {abs(full_acc - pruned_acc) * n_test:.0f} patients. It is noise, and the")
print("test set has no more claim to truth than the 5 folds of cross-validation that")
print("disagreed with it - the folds are averaged over far more data.")
print()
print("The honest summary: the two models are indistinguishable in accuracy, and one of")
print(f"them has {pruned.get_n_leaves()} leaves instead of {full.get_n_leaves()}. That is")
print("the win. Pruning bought readability at no measurable cost, which is exactly what it")
print("is supposed to do - it is a variance-reduction tool, not an accuracy-boosting one.")

## 1.6 Scale invariance — and the price you pay for it

A split asks `feature <= threshold`. Apply **any monotone transform** to that feature —
multiply by 1000, add 7, take the log — and the *ordering* of values is unchanged, so the
same rows land on the same side of a correspondingly transformed threshold. The tree is
**identical**.

This is not "trees are robust to scale". It is exact invariance, and it is why you will never
see a `StandardScaler` in front of a tree.

In [ ]:
Xtr_arr, Xte_arr = X_train.to_numpy(), X_test.to_numpy()

base = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(Xtr_arr, y_train)

# (a) affine rescale
affine = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(
    Xtr_arr * 1000 + 7, y_train)
# (b) a monotone NON-linear transform
shift = Xtr_arr.min(axis=0)
log_tr, log_te = np.log1p(Xtr_arr - shift + 1), np.log1p(Xte_arr - shift + 1)
logged = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(log_tr, y_train)

print("decision tree")
print("  predictions identical after  x*1000 + 7 :",
      np.array_equal(base.predict(Xte_arr), affine.predict(Xte_arr * 1000 + 7)))
print("  predictions identical after  log1p(x)   :",
      np.array_equal(base.predict(Xte_arr), logged.predict(log_te)))

lr_scaled = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr_arr, y_train)
lr_raw = LogisticRegression(max_iter=5000).fit(Xtr_arr, y_train)
print("\nlogistic regression, for contrast")
print("  predictions identical scaled vs raw     :",
      np.array_equal(lr_scaled.predict(Xte_arr), lr_raw.predict(Xte_arr)))
print()
print("What this buys you in practice:")
print("  - no scaler in the pipeline, so one less fitted step that could leak")
print("  - skewed features need no log transform")
print("  - ORDINAL categories can be integer-encoded directly (small<medium<large as 0,1,2)")
print("    because the tree only ever compares them with <=, respecting the order")
print()
print("What it does NOT excuse: nominal categories. Encoding {red, blue, green} as {0,1,2}")
print("still tells the tree that blue is 'between' red and green, which is nonsense.")

### The price: axis-aligned boundaries

Every split is a cut **perpendicular to one feature axis**. So the decision boundary is
always a union of axis-aligned rectangles. A diagonal boundary — the thing a linear model
draws with one coefficient — has to be approximated by a staircase.

In [ ]:
rng = np.random.default_rng(1)
n_diag = 2000
X_diag = rng.uniform(-3, 3, size=(n_diag, 2))
y_diag = (X_diag[:, 0] + X_diag[:, 1] > 0).astype(int)     # a perfect diagonal
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    X_diag, y_diag, test_size=0.3, random_state=RANDOM_STATE)

print(f"{'model':<24} {'test accuracy':>14} {'leaves':>8}")
print("-" * 48)
models = [("logistic regression", LogisticRegression()),
          ("tree, depth 3", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
          ("tree, depth 6", DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE)),
          ("tree, unlimited", DecisionTreeClassifier(random_state=RANDOM_STATE))]
fitted = []
for label, m in models:
    m.fit(Xd_tr, yd_tr)
    fitted.append((label, m))
    leaves = m.get_n_leaves() if hasattr(m, "get_n_leaves") else "-"
    print(f"{label:<24} {m.score(Xd_te, yd_te):>14.4f} {str(leaves):>8}")

# Draw the boundaries.
xx, yy = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (label, m) in zip(axes, [fitted[0], fitted[1], fitted[3]]):
    ax.contourf(xx, yy, m.predict(grid).reshape(xx.shape), alpha=0.3, levels=1)
    ax.scatter(Xd_te[:, 0], Xd_te[:, 1], c=yd_te, s=4, alpha=0.5)
    ax.set(title=f"{label}\ntest acc {m.score(Xd_te, yd_te):.3f}")
fig.tight_layout(); plt.show()

print("One coefficient beats 44 leaves. The tree is not wrong - it is spending enormous")
print("capacity approximating something a linear model expresses exactly.")
print()
print("The practical fix is a FEATURE, not a hyperparameter: give it x1 + x2 (or a rotation,")
print("or a ratio) and the tree needs one split. This is the mirror image of NB-02's XOR")
print("problem - each model family is blind to a different shape, and the cure is the same:")
print("engineer the feature that makes the structure visible.")

## 1.7 Regression trees

Everything transfers. Only two things change:

- **Impurity becomes variance** (equivalently MSE). A split is scored by how much it reduces
  the weighted variance of the target within the children.
- **A leaf predicts the mean** of its training samples, instead of the majority class.

The consequence is that a regression tree's output is a **step function** — piecewise
constant, with one value per leaf. That has a dramatic implication.

In [ ]:
rng = np.random.default_rng(2)
x_reg = np.linspace(0, 10, 120).reshape(-1, 1)
y_reg = 2 * x_reg.ravel() + 3 + rng.normal(0, 1, 120)      # truth: y = 2x + 3

tree_reg = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE).fit(x_reg, y_reg)
lin_reg = LinearRegression().fit(x_reg, y_reg)

smooth = np.linspace(0, 16, 400).reshape(-1, 1)
plt.scatter(x_reg, y_reg, s=12, alpha=0.5, label="training data (x from 0 to 10)")
plt.plot(smooth, tree_reg.predict(smooth), color="crimson", label="regression tree (depth 4)")
plt.plot(smooth, lin_reg.predict(smooth), color="steelblue", ls="--", label="linear regression")
plt.axvline(10, color="gray", ls=":", label="edge of training data")
plt.xlabel("x"); plt.ylabel("y"); plt.title("A tree predicts a staircase - and it is FLAT outside the data")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

query = np.array([[5.], [10.], [15.], [25.]])
print(f"{'x':>6} {'tree':>10} {'linear':>10} {'truth 2x+3':>12}")
print("-" * 42)
for xi, pt, pl in zip(query.ravel(), tree_reg.predict(query), lin_reg.predict(query)):
    print(f"{xi:>6.0f} {pt:>10.2f} {pl:>10.2f} {2*xi+3:>12.2f}")
print(f"\ntraining y ranged from {y_reg.min():.2f} to {y_reg.max():.2f}")
print()
print("A REGRESSION TREE CANNOT EXTRAPOLATE. Past the edge of the training data it returns")
print("the value of the last leaf, forever. At x=25 the truth is 53 and it says 23.")
print()
print("This is not a bug or a tuning failure - it is structural. A leaf predicts the mean of")
print("the training samples that fell in it, so no prediction can ever leave the range of the")
print("training targets.")
print()
print("It is also the direct opposite of NB-01's warning, where a linear model extrapolated")
print("confidently and wrongly. Trees fail SAFELY (they refuse to extrapolate) and linear")
print("models fail LOUDLY (they will happily predict a negative age). Neither is 'better' -")
print("you need to know which failure you have.")
print()
print("Practical consequence: never use a tree-based model for a trending time series")
print("without differencing or detrending first. It cannot predict a value it has not seen.")

---
# Part 2 - Worked example: breast cancer diagnosis

569 patients, 30 measurements from digitised images of a fine-needle aspirate. Predict
**malignant (0)** or **benign (1)**.

This is the right dataset for a tree because the deliverable is not only a prediction — it is
a rule set a clinician can read, argue with, and reject. We follow the Foundations workflow
and focus on what is tree-specific.

In [ ]:
# Baseline first, always.
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"baseline (always predict benign): accuracy {dummy.score(X_test, y_test):.4f}")
print(f"class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print()

# No scaler, no imputer - a tree needs neither here. That is the whole pipeline.
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [2, 3, 4, 5, 6, None],
        "min_samples_leaf": [1, 3, 5, 10, 20],
        "criterion": ["gini", "entropy"],
    },
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc", n_jobs=-1,
).fit(X_train, y_train)

print("best parameters:", grid.best_params_)
print(f"best CV ROC-AUC: {grid.best_score_:.4f}")

best_tree = grid.best_estimator_
print(f"\nchosen tree: {best_tree.get_n_leaves()} leaves, depth {best_tree.get_depth()}")

In [ ]:
# THE deliverable: the rules themselves.
print(export_text(best_tree, feature_names=feature_names, max_depth=3))
print("A clinician can audit every line of this. They can tell you that a threshold is")
print("implausible, or that a feature is measured unreliably at their lab - feedback you")
print("simply cannot get from a 500-tree ensemble.")
print()
print("Two things to notice while reading it:")
print()
print("1. Some splits have children that predict the SAME class. That is not a bug: the")
print("   split still reduced impurity (the two leaves have different class PROPORTIONS,")
print("   so different predicted probabilities), it just did not move either side across")
print("   the 50% line. Those splits matter for predict_proba and are dead weight for")
print("   predict - a reason to prune on the metric you actually care about.")
print()
print("2. 'truncated branch of depth N' means export_text hit our max_depth=3 display")
print("   limit, not the end of the tree. Raise it to print the whole thing.")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(best_tree, feature_names=feature_names, class_names=["malignant", "benign"],
          filled=True, rounded=True, fontsize=7, max_depth=2, ax=ax)
ax.set_title("The top of the fitted tree (first 2 levels)")
plt.tight_layout(); plt.show()

print("Reading a node: the split condition, gini, n samples reaching it, the class counts,")
print("and the majority class. Colour intensity shows purity.")

In [ ]:
test_pred = best_tree.predict(X_test)
test_proba = best_tree.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()
print("               predicted malignant   predicted benign")
print(f"  malignant  {tn:>18}   {fp:>17}")
print(f"  benign     {fn:>18}   {tp:>17}")
print()
print(classification_report(y_test, test_pred,
                            target_names=["malignant", "benign"], digits=3))
print(f"ROC-AUC {roc_auc_score(y_test, test_proba):.4f}")
print()
print(f"The costly error here is the {fp} malignant tumour(s) called benign - a missed")
print("cancer, not a false alarm. Accuracy weighs those the same as a benign tumour sent")
print("for an unnecessary biopsy, which is why accuracy is the wrong headline number.")
print()
print("For a screening tool you would move the threshold hard towards recall on the")
print("malignant class (NB-02 Part 3), accepting more false alarms to miss fewer cancers.")

### Feature importance, and why it needs a second opinion

`feature_importances_` is **MDI** (mean decrease in impurity): the total impurity reduction
each feature contributed, weighted by how many samples passed through those splits.

It is computed on the *training* data, and it is biased towards features with many possible
split points — continuous and high-cardinality features look important even when they are
noise. **Permutation importance** on held-out data measures something more honest: how much
does the score actually drop if I shuffle this column?

In [ ]:
mdi = pd.Series(best_tree.feature_importances_, index=feature_names)
perm = permutation_importance(best_tree, X_test, y_test, n_repeats=20,
                              random_state=RANDOM_STATE, scoring="roc_auc")
comparison = pd.DataFrame({
    "MDI_train": mdi,
    "permutation_test": perm.importances_mean,
}).sort_values("permutation_test", ascending=False)

print(comparison.head(8).round(4).to_string())
print(f"\nfeatures with MDI exactly 0 (never used in a split): "
      f"{int((mdi == 0).sum())} of {len(mdi)}")
print()
print("A single pruned tree uses only a handful of features, so most importances are exactly")
print("zero. That is NOT evidence those features are uninformative - many are strongly")
print("correlated with the ones chosen, and would have worked nearly as well.")
print()
print("This is the tree's interpretability trap: it tells you a sufficient rule, not the")
print("complete set of things that matter. Random forests (NB-04) spread importance across")
print("correlated features precisely because each tree sees a different feature subset.")

---
# Part 3 - Instability: the signature weakness

Here is the fact that should change how you present a tree.

**Refit the same tree on a slightly different sample of the same data and you can get a
completely different tree** — different root split, different features, different thresholds
— with essentially the same accuracy.

This is high **variance** in the bias-variance sense, and it is the direct consequence of
greedy splitting: two candidate splits can be nearly tied at the root, a handful of resampled
rows flips which one wins, and because everything below depends on that choice, the entire
tree below it changes.

In [ ]:
# Refit on 60 bootstrap resamples and record what each tree chooses at the root.
rng = np.random.default_rng(0)
# Fit on plain arrays throughout, so scoring must use arrays too - mixing a
# DataFrame fit with an array predict (or vice versa) triggers a feature-name warning.
Xtr_arr2, ytr_arr2 = X_train.to_numpy(), y_train.to_numpy()
Xte_arr2, yte_arr2 = X_test.to_numpy(), y_test.to_numpy()

roots, accuracies = {}, []
for _ in range(60):
    idx = rng.choice(len(Xtr_arr2), len(Xtr_arr2), replace=True)
    t = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(
        Xtr_arr2[idx], ytr_arr2[idx])
    root_feature = feature_names[t.tree_.feature[0]]
    roots[root_feature] = roots.get(root_feature, 0) + 1
    accuracies.append(t.score(Xte_arr2, yte_arr2))

accuracies = np.array(accuracies)
print(f"60 bootstrap resamples of the SAME training data, same hyperparameters\n")
print(f"distinct features chosen as the ROOT split: {len(roots)}")
for feature, count in sorted(roots.items(), key=lambda kv: -kv[1]):
    print(f"   {feature:<26} chosen {count:>2}/60 times")

print(f"\ntest accuracy across those 60 trees: "
      f"{accuracies.mean():.4f} +/- {accuracies.std():.4f} "
      f"(range {accuracies.min():.4f} to {accuracies.max():.4f})")
print()
print("Read those two blocks together, because the combination is the point:")
print("  the trees DISAGREE about which measurement matters most,")
print("  and they perform ABOUT THE SAME.")
print()
print("So 'the model says worst_perimeter is the key indicator' is not a finding. It is an")
print("artifact of which patients happened to be in your sample.")

In [ ]:
# The same instability, visible in the rules themselves.
print("Three trees, three resamples of the same data, first two levels each:\n")
rng = np.random.default_rng(7)
for i in range(3):
    idx = rng.choice(len(Xtr_arr2), len(Xtr_arr2), replace=True)
    t = DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE).fit(
        Xtr_arr2[idx], ytr_arr2[idx])
    print(f"--- resample {i+1}   (test accuracy {t.score(Xte_arr2, yte_arr2):.4f}) ---")
    print(export_text(t, feature_names=feature_names))

### What to do about it

| If you need | Do this |
|---|---|
| **A stable, accurate model** | Average many trees — a **random forest** (NB-04). Variance falls, and the accuracy usually rises with it. |
| **A stable, readable rule set** | Prune hard. Shallow trees are far more stable, because the top splits are the ones with large, decisive gains. |
| **Honest feature importance** | Never from one tree. Use a forest's importances, or permutation importance, and report a range. |
| **A certified-optimal small tree** | An optimal-tree solver (GOSDT, `pystreed`) — genuinely reproducible, at the cost of scale. |

### And this is exactly why ensembles exist

Look at what Part 3 has established:

- individual trees are **high variance** — they change a lot with the data
- but they are **not systematically wrong** — the average accuracy is decent

That combination is the textbook case for **averaging**. Independent-ish high-variance,
low-bias estimators averaged together keep the low bias and shed the variance. Bagging does
exactly this, and random forests add one more trick — decorrelating the trees by restricting
the features available at each split.

You now have the motivation for the next notebook, derived rather than asserted.

In [ ]:
# Averaging trees, measured. Same data, same depth, the only change is the count.
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
print(f"{'model':<40} {'CV ROC-AUC':>12}")
print("-" * 54)
single = cross_val_score(DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
                         X_train, y_train, cv=cv, scoring="roc_auc")
print(f"{'one tree (depth 4)':<40} {single.mean():>7.4f} +/-{single.std():<5.3f}")
for n in [10, 100]:
    f = cross_val_score(RandomForestClassifier(n_estimators=n, max_depth=4,
                                               random_state=RANDOM_STATE),
                        X_train, y_train, cv=cv, scoring="roc_auc")
    print(f"{f'random forest of {n} such trees':<40} {f.mean():>7.4f} +/-{f.std():<5.3f}")

print()
print("Higher mean AND a smaller standard deviation, from the identical base learner.")
print("Nothing about the individual trees improved - we only stopped trusting any one of")
print("them. That is the entire idea of NB-04.")
print()
print("The cost is the thing this notebook is about: you can print the depth-4 tree and")
print("hand it to a clinician. You cannot hand them a hundred of them.")

---
# Part 4 - Tough questions

Answer each **before** expanding it.

---

### Q1. An unpruned decision tree reaches 100% training accuracy on almost any dataset. Why is that guaranteed, and what is the one condition that breaks the guarantee?

<details><summary>Answer</summary>

The tree splits until every leaf is pure, and it can always keep going: with continuous
features and distinct values, there is always a threshold that isolates any single sample.
In the limit every leaf holds one training row, and a leaf predicts its own contents
perfectly.

**The condition that breaks it:** two rows with **identical feature values but different
labels**. No split can separate them, because splits only compare feature values — so that
leaf is irreducibly impure and training accuracy is capped below 1.

This is a genuinely useful diagnostic. If an unpruned tree does *not* reach 100%, you have
contradictory duplicates, which usually means either a data-entry problem or that your
features are missing something essential for those cases.

It also explains why training accuracy is completely uninformative for trees. The only
question is what happens on data it has not seen.

</details>

---

### Q2. Gini and entropy: when does the choice actually matter, and why is Gini the default?

<details><summary>Answer</summary>

**It almost never matters.** Part 1.2 measures it: the accuracy difference is a small
fraction of one standard deviation, and the two criteria pick the same root split in the
large majority of bootstrap resamples. Both are zero at purity, maximal at 50/50, and
strictly concave; they disagree only in a narrow band of class ratios.

**Gini is the default for speed.** Entropy needs a logarithm for every candidate split;
Gini needs a multiplication. A tree evaluates enormous numbers of candidate splits, so that
constant factor is real — and the resulting trees are nearly identical.

The one nuance worth knowing: entropy is very slightly more inclined to produce balanced
splits, because $-p\log p$ grows faster than $1-p^2$ near the extremes. In practice this is
swamped by the variance in Q7.

If you want a criterion that genuinely changes behaviour, look at **misclassification error**
— which is *not* used for growing, precisely because it is not strictly concave and often
gives zero gain for splits that clearly improve the tree (the XOR situation in 1.4). Gini and
entropy are chosen for their concavity, not their statistical interpretation.

</details>

---

### Q3. Why is every practical implementation greedy, when greedy is provably not optimal?

<details><summary>Answer</summary>

**Because finding the optimal tree is NP-hard** (Hyafil & Rivest, 1976). The search space of
trees is combinatorial in features × thresholds × depth, so exhaustive optimisation is
infeasible beyond toy problems.

Greedy growth costs roughly $O(n\,p\log n)$ per level and gives good trees in practice.

**The cost is real and Part 1.4 demonstrates it.** On XOR, every possible first split has a
gain of essentially zero, so one-step lookahead sees no reason to prefer either — yet a
depth-2 tree is perfect. Greedy cannot reason about a split that is worthless alone and
essential in combination.

**The standard mitigations:**

- **grow-then-prune** (1.5) — deferred correction: grow past the useless split, then cut back
- **ensembles** — many greedy trees on perturbed data, averaged (NB-04, NB-05)
- **optimal-tree solvers** — MIP/SAT formulations (GOSDT, `pystreed`) that are genuinely
  optimal for small, shallow trees. Worth knowing about when you need a certified-minimal
  model for regulatory audit.

</details>

---

### Q4. What exactly does `ccp_alpha` penalise, and why is the pruning path a *finite* list when alpha is continuous?

<details><summary>Answer</summary>

Cost-complexity pruning minimises

$$ R_\alpha(T) = R(T) + \alpha\,|T| $$

where $R(T)$ is the tree's total error and $|T|$ is its **number of leaves**. So $\alpha$ is
the price of a leaf: raise it and leaves must justify themselves with more error reduction.

**Why the path is finite:** as $\alpha$ increases continuously, the optimal subtree does not
change continuously — it stays *exactly the same* until $\alpha$ reaches the point where some
subtree's error reduction no longer covers its leaf cost, at which point that whole subtree
collapses to a leaf in one step. Between those critical values nothing changes.

So there is a **finite nested sequence** of optimal subtrees $T_0 \supset T_1 \supset \dots
\supset \{\text{root}\}$, and `cost_complexity_pruning_path` returns exactly the $\alpha$
values where the tree jumps. You then choose among that handful by cross-validation.

Contrast with Ridge's `alpha`, where the coefficient path *is* continuous — every tiny change
in alpha changes every coefficient a little. Here the model space is discrete.

</details>

---

### Q5. Why do trees need no feature scaling, and what does that let you get away with that other models do not?

<details><summary>Answer</summary>

A split is `feature <= threshold`, so only the **rank ordering** of a feature's values is
used. Apply any strictly monotone transform — multiply by 1000, add 7, take the log,
square-root — and the ordering is unchanged, so the same rows fall on the same side of a
correspondingly transformed threshold. The tree is **exactly identical**, not merely similar.
Part 1.6 verifies this.

**What you get away with:**

- no `StandardScaler` — one less fitted step in the pipeline, one less thing to leak
- no log-transforming skewed features
- **ordinal categories can be integer-encoded directly** — `small<medium<large` → 0,1,2 is
  correct, because `<=` respects the ordering. Linear models need this to be genuinely
  linear in the target; trees do not.
- outliers in the *features* barely matter — an extreme value is just "the largest", and it
  cannot drag a threshold the way it drags a least-squares coefficient

**What you do NOT get away with:** nominal categories. Encoding {red, blue, green} as
{0,1,2} still tells the tree that blue lies between red and green, so the only splits
available are `<= 0.5` and `<= 1.5` — which forces an arbitrary grouping. Use one-hot, or a
model with native categorical support (LightGBM, CatBoost, `HistGradientBoosting`).

Also note: outliers in the **target** of a regression tree still matter, because a leaf
predicts a mean.

</details>

---

### Q6. Your tree needs 44 leaves to approximate a boundary a logistic regression captures with two coefficients. What is going on, and what is the fix?

<details><summary>Answer</summary>

Every split is perpendicular to one feature axis, so the decision boundary is a union of
**axis-aligned rectangles**. A diagonal boundary like $x_1 + x_2 > 0$ is not expressible; it
can only be approximated by a staircase, and each step costs a split. Part 1.6 shows the tree
reaching 0.978 with 44 leaves where logistic regression gets 0.997 with one line.

**The fix is a feature, not a hyperparameter.** Give the tree `x1 + x2` and it needs a single
split. In general: rotations, sums, differences and ratios are cheap for you to compute and
structurally impossible for an axis-aligned learner to discover.

This is the exact mirror of NB-02's XOR problem, and the symmetry is the lesson: **every
model family is blind to some shape**, and the cure in both directions is to engineer the
feature that makes the structure visible. Linear models cannot see interactions unless you
multiply; trees cannot see rotations unless you add.

(Oblique trees, which split on linear combinations, exist and solve this directly — at the
cost of the readability that is the whole reason to use a tree.)

</details>

---

### Q7. Two people fit a tree on 95% samples of the same dataset and get different root splits. Who made a mistake?

<details><summary>Answer</summary>

**Neither.** This is the defining property of trees, and Part 3 quantifies it: 60 bootstrap
resamples of the same training data produce several different root features, with the top two
chosen roughly equally often — and all the trees score about the same on the test set.

**The mechanism:** at the root, two candidate splits are often nearly tied in impurity gain.
A handful of resampled rows flips which one wins. Because every split below is fitted
*conditional on* that choice, the entire tree changes. Greedy recursion turns a tiny
perturbation at the top into a completely different structure below.

**Why it matters:** it means "the model identified X as the key driver" is not a finding — it
is a statement about which rows happened to be in your sample. Never report a single tree's
structure as insight without checking stability.

**What to do:** prune hard (shallow trees are much more stable, because the top splits have
large decisive gains); use a forest's averaged importances, or permutation importance;
report a range rather than a ranking.

**And this is the argument for ensembles.** High variance + low bias + roughly-independent
errors is precisely the situation where averaging wins, which is what bagging and random
forests do.

</details>

---

### Q8. Why can a regression tree never predict a value outside the range of its training targets, and when does that matter enormously?

<details><summary>Answer</summary>

A leaf predicts the **mean of the training samples that reached it**. A mean of values drawn
from $[y_{\min}, y_{\max}]$ is itself inside that range, so **no prediction can ever leave
it**. Beyond the edge of the training data the tree returns the last leaf's value, flat,
forever. Part 1.7 shows the truth at $x{=}25$ being 53 while the tree says 23.

**When it matters enormously: any trending series.** Sales growing 10% a year, an ageing
population, an inflating price index — a tree-based model cannot predict next year's value if
it exceeds anything it has seen. It will confidently under-forecast forever, and the error
grows with the trend.

**The fixes:** difference or detrend the target first (model the *change*, which is
stationary), or model the trend separately and let the tree handle the residual, or use a
linear model — or a hybrid — for the extrapolating component.

**The flip side, which is genuinely a virtue:** trees fail *safely*. A linear model asked
about a 200-year-old patient returns a number with total confidence (NB-01 Part 6.3); a tree
returns something within the range of real observed outcomes. Bounded nonsense beats
unbounded nonsense. Neither is "better" — you need to know which failure mode you have.

</details>

---

### Q9. `feature_importances_` says a feature has importance 0.0. Does that mean it is uninformative?

<details><summary>Answer</summary>

**No.** It means the feature was **never chosen for a split**, which is a much weaker
statement. Part 2 shows a pruned tree using only a handful of the 30 features, with most
importances exactly zero on a dataset where nearly every feature is predictive.

**The most common reason is correlation.** If two features carry nearly the same information,
whichever wins at a node takes the entire impurity reduction, and the other never gets a
chance to be useful — its importance is 0 despite being individually excellent.

**Two further biases in MDI:**

1. It is computed on **training** data, so it rewards splits that helped memorise noise.
2. It is biased towards **high-cardinality and continuous** features, which offer more
   candidate thresholds and so more chances to reduce impurity by luck. A random continuous
   ID column can show meaningful MDI.

**What to do instead:** permutation importance on **held-out** data (what does the score
actually lose if I shuffle this column?), or a forest's averaged importances, which spread
credit across correlated features because each tree sees a different feature subset.

And remember what neither measures: **causation**. A high-importance feature may be a proxy,
a consequence of the target, or a confounder.

</details>

---

### Q10. `max_depth`, `min_samples_leaf`, `max_leaf_nodes`, `ccp_alpha` all limit complexity. Which should you reach for, and how do they differ?

<details><summary>Answer</summary>

They constrain along different axes, and the differences are practical:

- **`max_depth`** — a uniform cap. Blunt: it forces the same limit on a dense region with
  thousands of samples and a sparse one with twelve. Good first knob because it is easy to
  reason about.
- **`min_samples_leaf`** — no leaf may be smaller than this. **Usually the most useful
  single parameter**, because it directly targets the actual failure mode: rules fitted to a
  handful of rows. It adapts naturally — deep branches are allowed where data is dense.
- **`max_leaf_nodes`** — a total budget, grown **best-first** rather than depth-first, so the
  tree spends its leaves where they buy the most. Often better than `max_depth` at equal size.
- **`ccp_alpha`** — the only *post*-pruning option, and the only principled one: grow fully,
  then remove subtrees that do not pay for themselves. It avoids the pre-pruning trap of
  killing a split whose value only appears one level lower (the XOR problem).

**In practice:** tune `min_samples_leaf` and `ccp_alpha`, and use `max_depth` mainly to bound
runtime or to force readability. Note they interact, so tune jointly rather than one at a
time.

Also worth knowing: inside a **random forest** you usually *want* deep, unpruned trees. Bias
is what averaging cannot fix and variance is what it can, so individual trees should be low-
bias/high-variance and let the ensemble handle the rest.

</details>

---

### Q11. Trees handle mixed feature types and missing values more gracefully than most models. Where does that actually stop?

<details><summary>Answer</summary>

**What genuinely works:**

- **Numeric features of any scale or skew**, mixed freely (Q5).
- **Ordinal categories**, integer-encoded — `<=` respects the order.
- **Missing values**, in modern implementations. `HistGradientBoosting*`, LightGBM and
  XGBoost learn a default direction for NaN at each split, so missingness becomes a signal
  rather than a problem. Classic CART used *surrogate splits* — a backup feature correlated
  with the primary one.

**Where it stops:**

- **sklearn's `DecisionTreeClassifier` gained NaN support relatively recently** and it is
  limited — do not assume it; check your version, and impute in a Pipeline if unsure.
- **Nominal categories still need encoding** (Q5). One-hot on a high-cardinality column is
  actively harmful for trees: each binary column can only split off one level at a time, so
  a genuinely useful grouping ("these 12 postcodes behave alike") needs 12 stacked splits and
  the tree is unlikely to find it. This is why LightGBM and CatBoost implement native
  categorical splitting, and why target encoding is popular here.
- **Text and images** — no. Trees need tabular features; you must extract them first.
- **Missing-not-at-random** still biases the model. Handling NaN mechanically is not the same
  as handling it *correctly* (Foundations Part 5.1).

</details>

---

### Q12. When would you ship a single decision tree, knowing a random forest will almost certainly score better?

<details><summary>Answer</summary>

This is a genuine engineering judgement, not a trick question. Ship the single tree when:

- **The rule set is the deliverable.** Regulated lending, clinical protocols, fraud rules
  that a human team must execute. "Print it and put it on the wall" is a real requirement,
  and an ensemble cannot satisfy it at any accuracy.
- **A domain expert must be able to reject it.** A clinician can look at a threshold and say
  "that assay is unreliable below 0.2 at our lab". You cannot get that feedback on 500 trees,
  and that feedback is often worth more than the accuracy you gave up.
- **You must explain individual decisions**, exactly, cheaply, and identically every time —
  "declined because income < X **and** existing debt > Y". No SHAP approximation, no
  post-hoc rationalisation.
- **The inference budget is extreme** — a handful of comparisons, no floating-point
  arithmetic needed, trivially portable to SQL, a spreadsheet, or embedded hardware.
- **The accuracy gap is small and the stakes of complexity are high.** Measure it. If a
  pruned tree is within a point of the forest, the forest may not be worth the operational
  burden.

The honest framing: you are trading measured accuracy for auditability, and you should
**know the size of the trade** rather than assume it. Always fit the forest too, so you can
state what interpretability cost.

Rudin (2019), in Part 6, argues this position much more strongly — that for high-stakes
decisions we should build interpretable models rather than explain black boxes after the
fact.

</details>

---

## Coding challenges

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 1
# Extend the from-scratch tree in Part 1.3 to REGRESSION.
#
# Replace Gini with variance, and make each leaf predict the mean of its
# samples. Verify against DecisionTreeRegressor, and confirm the predictions
# are piecewise constant.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 1: one solution ------------------------------------------
def variance_impurity(y):
    """The regression analogue of Gini: variance (equivalently MSE about the mean)."""
    return 0.0 if len(y) == 0 else float(np.var(y))


def best_split_regression(X, y):
    n, n_features = X.shape
    parent = variance_impurity(y)
    best = (0.0, None, None)
    for f in range(n_features):
        values = np.unique(X[:, f])
        if len(values) < 2:
            continue
        for t in (values[:-1] + values[1:]) / 2:
            mask = X[:, f] <= t
            n_left = mask.sum()
            if n_left == 0 or n_left == n:
                continue
            weighted = (n_left / n) * variance_impurity(y[mask]) + \
                       ((n - n_left) / n) * variance_impurity(y[~mask])
            if parent - weighted > best[0]:
                best = (parent - weighted, f, t)
    return best


def build_regression_tree(X, y, depth=0, max_depth=3):
    if depth >= max_depth or len(y) < 2:
        return {"leaf": True, "prediction": float(np.mean(y)), "n": len(y)}
    gain, feature, threshold = best_split_regression(X, y)
    if feature is None or gain <= 1e-12:
        return {"leaf": True, "prediction": float(np.mean(y)), "n": len(y)}
    mask = X[:, feature] <= threshold
    return {"leaf": False, "feature": feature, "threshold": threshold,
            "left":  build_regression_tree(X[mask],  y[mask],  depth + 1, max_depth),
            "right": build_regression_tree(X[~mask], y[~mask], depth + 1, max_depth)}


def predict_one_regression(node, x):
    """Walk to a leaf; the leaf holds a mean instead of a majority class."""
    while not node["leaf"]:
        node = node["left"] if x[node["feature"]] <= node["threshold"] else node["right"]
    return node["prediction"]


rng = np.random.default_rng(3)
x_c1 = np.sort(rng.uniform(0, 10, 200)).reshape(-1, 1)
y_c1 = np.sin(x_c1.ravel()) * 5 + rng.normal(0, 0.5, 200)

my_reg = build_regression_tree(x_c1, y_c1, max_depth=3)
mine = np.array([predict_one_regression(my_reg, x) for x in x_c1])
sk_reg = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE).fit(x_c1, y_c1)

print(f"from scratch MAE : {mean_absolute_error(y_c1, mine):.6f}")
print(f"scikit-learn MAE : {mean_absolute_error(y_c1, sk_reg.predict(x_c1)):.6f}")
print(f"predictions match: {np.allclose(mine, sk_reg.predict(x_c1))}")
print(f"\ndistinct predicted values: {len(np.unique(np.round(mine, 8)))} "
      f"for {len(x_c1)} inputs  <- piecewise constant, one value per leaf")

plt.scatter(x_c1, y_c1, s=8, alpha=0.4, label="data")
plt.step(x_c1.ravel(), mine, where="mid", color="crimson", label="our regression tree")
plt.xlabel("x"); plt.ylabel("y"); plt.title("Variance-splitting reproduces sklearn exactly")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 2
# Show that MDI feature importance is biased towards high-cardinality features.
#
# Build a dataset where the target depends ONLY on a binary feature, then add
# a pure-noise continuous column and a pure-noise high-cardinality ID column.
# Compare MDI against permutation importance on held-out data.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 2: one solution ------------------------------------------
rng = np.random.default_rng(11)
n_c2 = 2000

signal = rng.integers(0, 2, n_c2)                      # the ONLY real driver
noise_continuous = rng.normal(size=n_c2)               # pure noise, many split points
noise_id = rng.permutation(n_c2).astype(float)         # pure noise, every value unique
noise_binary = rng.integers(0, 2, n_c2).astype(float)  # pure noise, 1 split point

X_c2 = pd.DataFrame({"real_signal": signal.astype(float),
                     "noise_continuous": noise_continuous,
                     "noise_unique_id": noise_id,
                     "noise_binary": noise_binary})
y_c2 = (rng.random(n_c2) < np.where(signal == 1, 0.85, 0.15)).astype(int)

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_c2, y_c2, test_size=0.3, stratify=y_c2, random_state=RANDOM_STATE)

deep = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(Xc_tr, yc_tr)
perm = permutation_importance(deep, Xc_te, yc_te, n_repeats=20,
                              random_state=RANDOM_STATE, scoring="accuracy")

result = pd.DataFrame({
    "MDI_train": deep.feature_importances_,
    "permutation_test": perm.importances_mean,
}, index=X_c2.columns).sort_values("MDI_train", ascending=False)
print("UNPRUNED tree (the regime where the bias is worst)")
print(result.round(4).to_string())
print(f"\ntrain accuracy {deep.score(Xc_tr, yc_tr):.4f}   "
      f"test accuracy {deep.score(Xc_te, yc_te):.4f}")
print()
print("MDI hands large importance to the two pure-noise continuous columns - they offer")
print("thousands of candidate thresholds, so deep in the tree they can always find a split")
print("that separates a few training rows by luck.")
print()
print("Permutation importance on HELD-OUT data is not fooled: shuffling them costs the")
print("model nothing, because they carry no real information.")
print()
print("Note noise_binary, which has only ONE possible split point, gets far less MDI than")
print("the continuous noise - that is the cardinality bias isolated, since both are equally")
print("uninformative.")

shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Xc_tr, yc_tr)
print("\nPRUNED to depth 3, MDI:")
print(pd.Series(shallow.feature_importances_, index=X_c2.columns).round(4).to_string())
print("\nPruning largely fixes it - the noise features never get deep enough to exploit")
print("their cardinality. Another reason not to interpret an unpruned tree.")

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 3
# Quantify the interpretability/accuracy trade-off from Q12.
#
# For depths 1..12, plot CV accuracy against the number of leaves, and mark
# the random forest's accuracy as a horizontal line. Then answer: what is the
# smallest tree within 1 percentage point of the forest?
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 3: one solution ------------------------------------------
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
forest_score = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    X_train, y_train, cv=cv, scoring="accuracy").mean()

rows = []
for depth in range(1, 13):
    s = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE),
                        X_train, y_train, cv=cv, scoring="accuracy")
    leaves = DecisionTreeClassifier(max_depth=depth,
                                    random_state=RANDOM_STATE).fit(X_train, y_train).get_n_leaves()
    rows.append({"depth": depth, "leaves": leaves,
                 "cv_accuracy": s.mean(), "cv_std": s.std()})
tradeoff = pd.DataFrame(rows)
print(tradeoff.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print(f"\nrandom forest (300 trees) CV accuracy: {forest_score:.4f}")

within_1pt = tradeoff[tradeoff["cv_accuracy"] >= forest_score - 0.01]
if len(within_1pt):
    winner = within_1pt.nsmallest(1, "leaves").iloc[0]
    print(f"\nsmallest tree within 1 point of the forest: depth {int(winner['depth'])}, "
          f"{int(winner['leaves'])} leaves, accuracy {winner['cv_accuracy']:.4f}")
    print(f"  interpretability cost: {forest_score - winner['cv_accuracy']:+.4f} accuracy")
    print(f"  what you gain: {int(winner['leaves'])} readable rules instead of 300 trees")
else:
    print("\nno tree comes within 1 point of the forest on this data - the ensemble is")
    print("genuinely buying accuracy here, and you would have to justify the trade.")

plt.errorbar(tradeoff["leaves"], tradeoff["cv_accuracy"], yerr=tradeoff["cv_std"],
             marker="o", ms=4, capsize=3, label="single tree")
plt.axhline(forest_score, ls="--", color="crimson", label="random forest (300 trees)")
plt.xlabel("number of leaves (model complexity)"); plt.ylabel("CV accuracy")
plt.title("What interpretability costs, measured")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

print("\nRun this on every project where someone asks for an interpretable model. It turns")
print("'trees are less accurate' from an assumption into a number you can put in a decision.")

---
# Part 5 - Five datasets to practise on

| # | Dataset | Rows × cols | The skill it forces | Difficulty |
|---|---|---|---|---|
| 1 | **Breast cancer** | 569 × 30 | Pruning, reading rules | ★☆☆☆☆ |
| 2 | **Titanic** | 1,309 × 13 | Mixed types, missing values, a genuinely readable tree | ★★☆☆☆ |
| 3 | **Car evaluation** | 1,728 × 6 | All-ordinal features — where trees shine | ★★★☆☆ |
| 4 | **Bike sharing** | 17,379 × 12 | **Regression trees and the extrapolation trap** | ★★★★☆ |
| 5 | **Adult / census** | 48,842 × 14 | High-cardinality categoricals, depth vs accuracy | ★★★★☆ |

In [ ]:
from sklearn.datasets import fetch_openml

CATALOGUE = [
    ("Breast cancer",   lambda: load_breast_cancer(as_frame=True)),
    ("Titanic",         lambda: fetch_openml(name="titanic", version=1, as_frame=True)),
    ("Car evaluation",  lambda: fetch_openml(name="car", version=3, as_frame=True)),
    ("Bike sharing",    lambda: fetch_openml(name="Bike_Sharing_Demand", version=2,
                                             as_frame=True)),
    ("Adult (census)",  lambda: fetch_openml(name="adult", version=2, as_frame=True)),
]

print(f"{'dataset':<18} {'rows':>7} {'cols':>6} {'NaN':>7} {'text':>6}  target")
print("-" * 66)
for label, loader in CATALOGUE:
    try:
        b = loader()
        Xd = b.data
        n_text = len(Xd.select_dtypes(exclude="number").columns)
        print(f"{label:<18} {Xd.shape[0]:>7} {Xd.shape[1]:>6} "
              f"{int(Xd.isna().sum().sum()):>7} {n_text:>6}  {b.target.name}")
    except Exception as exc:
        print(f"{label:<18} unavailable - {type(exc).__name__}: {str(exc)[:32]}")

### 1. Breast cancer — pruning and readability

Already used in Part 2. Go further:

1. Tune `ccp_alpha` and `min_samples_leaf` **jointly** with `GridSearchCV`. Do they interact?
2. Find the smallest tree within 1 point of the best (Challenge 3) and write its rules out as
   English sentences.
3. A missed malignancy is far worse than a false alarm. Set `class_weight` and re-tune —
   what happens to tree *shape*, not just accuracy?

---

### 2. Titanic — the classic, for good reason

```python
bunch = fetch_openml(name="titanic", version=1, as_frame=True)
X, y = bunch.data, bunch.target        # survived: '0' / '1'
```

Mixed types, real missing values (`age` especially), and a tree that produces genuinely
memorable rules.

1. `name`, `ticket`, `cabin`, `boat`, `body` are identifiers or **post-outcome** fields.
   `boat` and `body` in particular leak the answer — find them and explain why before fitting.
2. Build a `ColumnTransformer` for the rest. Impute `age` — and add a missingness indicator;
   for whom is age missing, and is that informative?
3. Fit a depth-3 tree and read the rules aloud. They should reproduce something you already
   know about the disaster.
4. Extract a title (Mr/Mrs/Miss/Master) from `name`. Does it beat `age` + `sex`?

**Good result:** ~0.80 accuracy. **The trap:** `boat` — near-perfect accuracy, zero value.

---

### 3. Car evaluation — all-ordinal, tree-friendly

```python
bunch = fetch_openml(name="car", version=3, as_frame=True)
```

Six features, all ordinal (`low < med < high < vhigh`), four classes. The purest illustration
of Q5.

1. Encode with `OrdinalEncoder` **in the right order** (not alphabetical — check!) and compare
   against `OneHotEncoder`. Which wins, and why should ordinal win here?
2. The relationship is close to a deterministic rule set. How deep must the tree go to reach
   ~100%? Is that overfitting, or is it learning the actual rules?
3. Compare against logistic regression. This is a case where a tree should win decisively —
   explain why in terms of interactions.

---

### 4. Bike sharing — regression, and the extrapolation trap

```python
bunch = fetch_openml(name="Bike_Sharing_Demand", version=2, as_frame=True)
```

Hourly rental counts. **This one is designed to bite.**

1. Fit a `DecisionTreeRegressor` with a random split. Note the score.
2. Now split by **time** (train on early months, test on later). The score collapses. Explain
   it with Q8 — ridership *grew* over the period, so later demand exceeds anything in
   training and the tree literally cannot predict it.
3. Fix it: model the trend separately, or difference the target, then let the tree handle the
   seasonal/weather residual. Measure the improvement.
4. Compare against linear regression on the same time split. Which extrapolates, and which is
   better on the seasonal structure? Consider combining them.

**The lesson:** a model can be excellent under one validation scheme and useless under the
one that matches deployment.

---

### 5. Adult / census — categoricals at scale

```python
bunch = fetch_openml(name="adult", version=2, as_frame=True)
```

1. `native-country` has ~40 levels. One-hot it, then read Q11 again and explain why the tree
   struggles to use it well.
2. Compare one-hot against `HistGradientBoostingClassifier` with `categorical_features=` —
   how much does native categorical handling buy?
3. Plot CV accuracy against `max_depth` from 1 to 30. Where does it peak, and how big is the
   tree there?
4. Fit a depth-4 tree and check: does its rule set encode anything you would be uncomfortable
   deploying? (Look at which demographic features it splits on first.)

---
# Part 6 - Reading the literature

New to papers? `linear_regression_zero_to_hero.ipynb` **Part 9.1** covers Keshav's three-pass
method. First pass is 5–10 minutes.

## Start here

**1. [Stop Explaining Black Box Machine Learning Models for High Stakes Decisions and Use Interpretable Models Instead](https://arxiv.org/abs/1811.10154)** —
Cynthia Rudin, *Nature Machine Intelligence*, 2019. **Free.**
> The strongest available argument for Q12. Rudin's claim is that the accuracy/interpretability
> trade-off is largely a **myth** on structured data, and that post-hoc explanations of black
> boxes are unreliable in exactly the settings where they matter most. Read it before your
> next "we need SHAP for this" conversation. Genuinely readable, and it will change how you
> argue about model choice.

**2. [Induction of Decision Trees](https://link.springer.com/article/10.1007/BF00116251)** —
J.R. Quinlan, *Machine Learning* 1(1):81-106, 1986.
> The ID3 paper — entropy and information gain, from the source. Short, and unusually clear
> about *why* a criterion is chosen rather than just what it is.

**3. [Classification and Regression Trees](https://doi.org/10.1201/9781315139470)** —
Breiman, Friedman, Olshen & Stone, 1984. *(A book, not a paper.)*
> **CART.** Gini, cost-complexity pruning, surrogate splits for missing values, regression
> trees — essentially every default in `sklearn.tree` traces to this book. Read chapters 2–3
> for the pruning derivation behind Q4.

## The paper behind each section

| Section | Source | Free? |
|---|---|---|
| 1.2 — entropy / information gain | Quinlan, *Induction of Decision Trees*, **1986**; refined as C4.5, **1993** | 🔍 |
| 1.2, 1.5, 1.7 — Gini, pruning, regression trees | **Breiman, Friedman, Olshen & Stone**, *CART*, **1984** | 🔍 |
| 1.4 — why greedy | **Hyafil & Rivest**, *Constructing Optimal Binary Decision Trees is NP-Complete*, Information Processing Letters 5(1), **1976** | 🔍 |
| 1.4 — optimal trees, modern | **Bertsimas & Dunn**, *Optimal Classification Trees*, Machine Learning 106, **2017**; **Lin et al.**, *Generalized and Scalable Optimal Sparse Decision Trees* (GOSDT), ICML **2020** — [arXiv](https://arxiv.org/abs/2006.08690) | ✅ |
| 1.6 — oblique (non-axis-aligned) trees | **Murthy, Kasif & Salzberg**, *A System for Induction of Oblique Decision Trees*, JAIR 2, **1994** — [arXiv](https://arxiv.org/abs/cs/9408103) | ✅ |
| Part 3 — instability, and the case for averaging | **Breiman**, *Bagging Predictors*, Machine Learning 24(2), **1996**; *Heuristics of Instability and Stabilization in Model Selection*, Ann. Statist. 24(6), **1996** | 🔍 |
| Q9 — MDI's cardinality bias | **Strobl, Boulesteix, Zeileis & Hothorn**, *Bias in Random Forest Variable Importance Measures*, BMC Bioinformatics 8:25, **2007** — [link](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-8-25) | ✅ |
| Q12 — interpretability | **Rudin**, **2019** — [arXiv](https://arxiv.org/abs/1811.10154) | ✅ |
| Q12 — the counter-position | **Lundberg & Lee**, *A Unified Approach to Interpreting Model Predictions* (SHAP), NeurIPS **2017** — [arXiv](https://arxiv.org/abs/1705.07874) | ✅ |

**Legend:** ✅ free at the link · 🔍 no reliable free link — search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Rudin (2019).** It is the paper most likely to change a decision you actually make, and it
is precisely about the trade-off this entire notebook is built around.

---
# Appendix

Foundations has the general error table and pre-ship checklist. These are tree-specific.

| Symptom | Cause | Fix |
|---|---|---|
| Training accuracy exactly 1.0000 | Unpruned tree — this is the default | Not a bug. Prune, and ignore training score entirely |
| Test accuracy much worse than train | Overfitting | `min_samples_leaf`, `ccp_alpha`, `max_depth` |
| Refitting gives a totally different tree | **Normal** (Part 3) | Prune harder, or use a forest |
| `feature_importances_` mostly zeros | Correlated features; only one gets used | Permutation importance, or a forest |
| A random ID column looks important | MDI cardinality bias (Q9) | Permutation importance on held-out data; drop IDs |
| Regression predictions flat outside the data range | Structural (Q8) | Detrend, difference, or use a linear component |
| Ordinal category behaving oddly | `OrdinalEncoder` used alphabetical order | Pass explicit `categories=` in the right order |
| High-cardinality one-hot performing badly | One binary column splits off one level at a time (Q11) | Native categorical support, or target encoding |
| `could not convert string to float` | Nominal categories unencoded | `OneHotEncoder` via `ColumnTransformer` |
| Deep tree is very slow to fit | Exhaustive threshold search | `max_leaf_nodes`, or `HistGradientBoosting*` (binned) |

## Checklist for shipping a decision tree

Everything in the Foundations checklist, plus:

- [ ] Did I prune — and choose the pruning parameter by **cross-validation**, not by eye?
- [ ] Did I check stability by refitting on resamples before quoting any rule as insight?
- [ ] Did I fit a random forest too, so I know what interpretability cost me?
- [ ] Am I using permutation importance rather than MDI for any claim about what matters?
- [ ] For regression: are production inputs inside the training range? Is the target trending?
- [ ] Are ordinal categories encoded in the **correct** order?
- [ ] Have I actually printed the rules and read them, looking for implausible thresholds?
- [ ] If the rule set is the deliverable, has a domain expert reviewed it?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `random_forest_zero_to_hero.ipynb` | **The direct sequel.** Part 3 showed trees are high-variance and roughly unbiased — exactly the case where averaging wins. |
| `gradient_boosting_zero_to_hero.ipynb` | The other way to combine trees: sequential correction rather than averaging. Usually the strongest tabular model. |
| `explainable_ai_zero_to_hero.ipynb` | Once you leave a single tree behind, how do you explain anything? Picks up the Rudin-vs-SHAP argument from Part 6. |

See [`ZERO_TO_HERO_PLAN.md`](../ZERO_TO_HERO_PLAN.md) for the full roster and status.